# Unified Perception — OWL-SAM Adapter Training

Trains the `OwlToSamAdapter` that bridges OWL-ViT-B encoder output `(B, 576, 768)` to SAM-compatible features `(B, 256, 64, 64)`.

**Phase B:** Architecture validation with synthetic features (real OWL/SAM encoders in Phase C).

**GPU:** T4 (free Colab) is sufficient. Enable it via `Runtime → Change runtime type → T4 GPU`.

## 1. Check GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Clone Repo

In [ ]:
!git clone https://github.com/sutharsan-311/unified-perception.git
%cd unified-perception
!ls

## 3. Install Dependencies

In [ ]:
!pip install -q torchvision Pillow

## 4. Download COCO val2017 (~780 MB)

We use `val2017` (5000 images) for both train and validation since it's much smaller than `train2017` (18 GB). Phase B uses synthetic features anyway, so the actual image content doesn't affect training quality.

In [ ]:
import os
os.makedirs('data/coco', exist_ok=True)

# Download val2017 (~780 MB)
!wget -q --show-progress -O data/coco/val2017.zip http://images.cocodataset.org/zips/val2017.zip
!unzip -q data/coco/val2017.zip -d data/coco/
!rm data/coco/val2017.zip

# Use val2017 as both splits
!ln -s val2017 data/coco/train2017 2>/dev/null || true

print(f"Images available: {len(list(__import__('pathlib').Path('data/coco/val2017').glob('*.jpg')))}")

## 5. Verify Adapter Shape

In [ ]:
import sys
sys.path.insert(0, '.')

from nanosam.adapter import OwlToSamAdapter

adapter = OwlToSamAdapter()
print(f"Parameters:      {adapter.get_parameter_count():,}")
print(f"Memory (FP32):   {adapter.get_parameter_count() * 4 / 1e6:.2f} MB")

# Shape check
device = 'cuda' if torch.cuda.is_available() else 'cpu'
x = torch.randn(2, 576, 768).to(device)
adapter = adapter.to(device)
out = adapter(x)
print(f"\nInput:  {x.shape}")
print(f"Output: {out.shape}  (expected [2, 256, 64, 64])")
assert out.shape == (2, 256, 64, 64)
print("✓ Shape OK")

## 6. Train

Uses `COCO_SMALL` preset tuned for Colab T4:
- 1000 images, 5 epochs, batch size 4, FP16
- ~15–20 min on T4

In [ ]:
!python train.py \
  --dataset-dir data/coco \
  --num-images 1000 \
  --validation-size 100 \
  --batch-size 4 \
  --num-epochs 5 \
  --learning-rate 1e-3 \
  --fp16 \
  --device cuda \
  --checkpoint-dir checkpoints \
  --num-workers 2 \
  --log-interval 10

## 7. Check Checkpoints

In [ ]:
!ls -lh checkpoints/

## 8. Save to Google Drive (optional)

Mount Drive to persist checkpoints across sessions.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r checkpoints/ /content/drive/MyDrive/unified-perception-checkpoints/
# print('Saved to Google Drive!')

## 9. Load & Verify Best Checkpoint

In [ ]:
import glob

checkpoints = sorted(glob.glob('checkpoints/*.pth'))
print('Saved checkpoints:')
for ck in checkpoints:
    print(f'  {ck}')

# Load best checkpoint
best_ckpt = 'checkpoints/adapter_phase_b_best.pth'
if __import__('os').path.exists(best_ckpt):
    ck = torch.load(best_ckpt, map_location=device, weights_only=False)
    print(f'\nBest checkpoint:')
    print(f'  Epoch:          {ck["epoch"]}')
    print(f'  Best val loss:  {ck["best_val_loss"]:.6f}')
else:
    print('Best checkpoint not saved yet (only saved when val loss improves mid-epoch).')
    final_ckpt = 'checkpoints/adapter_phase_b_final.pth'
    if __import__('os').path.exists(final_ckpt):
        ck = torch.load(final_ckpt, map_location=device, weights_only=False)
        print(f'Final checkpoint loaded — best val loss: {ck["best_val_loss"]:.6f}')